<a href="https://colab.research.google.com/github/mrpunith-1920/gnn-fraud-detection/blob/main/04_gnn_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.0 MB/s  0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch
path = '/content/drive/MyDrive/gnn-project/data/elliptic_bitcoin_dataset/'
data = torch.load(path + 'elliptic_graph.pt', weights_only=False)
print(data)

Mounted at /content/drive
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timestep=[203769], train_mask=[203769], test_mask=[203769])


In [2]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.6 MB/s  0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
path = '/content/drive/MyDrive/gnn-project/data/elliptic_bitcoin_dataset/'
data = torch.load(path + 'elliptic_graph.pt', weights_only=False)
print(data)

Mounted at /content/drive
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timestep=[203769], train_mask=[203769], test_mask=[203769])


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using:", device)
data = data.to(device)

Using: cuda


In [5]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GCNConv, GATConv

class GNN(torch.nn.Module):
    def __init__(self, in_dim, hidden=64, kind='sage'):
        super().__init__()
        Conv = {'gcn': GCNConv, 'sage': SAGEConv, 'gat': GATConv}[kind]
        self.c1 = Conv(in_dim, hidden)
        self.c2 = Conv(hidden, hidden)
        self.out = torch.nn.Linear(hidden, 2)

    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        x = F.relu(self.c2(x, edge_index))
        return self.out(x)

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

def train_gnn(kind, epochs=200, hidden=64, lr=0.005, seed=42):
    torch.manual_seed(seed)
    model = GNN(data.num_features, hidden, kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    n_fraud = (data.y[data.train_mask] == 1).sum().item()
    n_legit = (data.y[data.train_mask] == 0).sum().item()
    weights = torch.tensor([1.0, n_legit / n_fraud]).to(device)

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=weights)
        loss.backward()
        opt.step()
        if epoch % 50 == 0:
            print(f"epoch {epoch}, loss {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        prob = F.softmax(out, dim=1)[:, 1][data.test_mask].cpu().numpy()
    y_true = data.y[data.test_mask].cpu().numpy()
    pred = (prob > 0.5).astype(int)

    row = {
        'Model': kind.upper(),
        'Precision': precision_score(y_true, pred),
        'Recall': recall_score(y_true, pred),
        'F1': f1_score(y_true, pred),
        'PR-AUC': average_precision_score(y_true, prob),
        'ROC-AUC': roc_auc_score(y_true, prob),
    }
    return model, row

In [7]:
model_sage, row_sage = train_gnn('sage')
print(row_sage)

epoch 0, loss 0.7150
epoch 50, loss 0.1140
epoch 100, loss 0.0571
epoch 150, loss 0.0388
{'Model': 'SAGE', 'Precision': 0.7659817351598174, 'Recall': 0.6195752539242844, 'F1': 0.6850433894844309, 'PR-AUC': np.float64(0.6646099117482652), 'ROC-AUC': np.float64(0.9150237125535101)}


In [8]:
model_gcn, row_gcn = train_gnn('gcn')
print(row_gcn)

epoch 0, loss 0.7042
epoch 50, loss 0.2359
epoch 100, loss 0.1615
epoch 150, loss 0.1261
{'Model': 'GCN', 'Precision': 0.5250501002004008, 'Recall': 0.4838411819021237, 'F1': 0.5036040365209035, 'PR-AUC': np.float64(0.46954505274685693), 'ROC-AUC': np.float64(0.8507825287794284)}


In [10]:
import pandas as pd

baseline = pd.DataFrame([
    {'Model': 'Random Forest', 'Precision': 0.989, 'Recall': 0.683, 'F1': 0.808, 'PR-AUC': 0.794, 'ROC-AUC': 0.931},
    {'Model': 'XGBoost',       'Precision': 0.867, 'Recall': 0.733, 'F1': 0.794, 'PR-AUC': 0.800, 'ROC-AUC': 0.926},
])

all_results = pd.concat([baseline, pd.DataFrame([row_sage, row_gcn])], ignore_index=True).round(3)
all_results.to_csv(path + 'results_so_far.csv', index=False)
all_results

,Model,Precision,Recall,F1,PR-AUC,ROC-AUC
0,Random Forest,0.989,0.683,0.808,0.794,0.931
1,XGBoost,0.867,0.733,0.794,0.800,0.926
2,SAGE,0.766,0.620,0.685,0.665,0.915
3,GCN,0.525,0.484,0.504,0.470,0.851


What this means
Your code isn’t broken. This ranking (tree models above GraphSAGE, above GCN) matches what published papers report on Elliptic, including the original dataset paper.
The Elliptic features already include neighbor information. 72 of the 165 features are aggregated from each transaction’s neighbors. The tree models therefore get much of the graph signal for free, and the GNN adds less.
GCN is weakest because it averages a node with its neighbors, which blurs the node’s own features. GraphSAGE keeps the node’s own features separate from the neighbor average, so it does better.
The ROC-AUC gap is small (0.915 vs 0.931 for GraphSAGE). The GNN ranks transactions almost as well, but at the default 0.5 cut-off it picks a worse precision/recall balance. Threshold tuning can help.

For your report, say it honestly: “On Elliptic, tree-based baselines outperformed GNNs because the features already encode neighborhood information. The GNN’s value lies in the graph-based explanations, and in hybrid models.” Examiners respect honest analysis more than inflated results.